# G14 — Atelier 1 : calcul du KPI de décision K1

Décision : date de début du renfort des saisonniers du SDIS 33 (question Q2). Territoire de référence : Gironde (33), campagnes 2021-2025.
Sources : BDIFF (feux, départements 33 et 40) et Code officiel géographique 2026 de l'Insee. Le notebook s'exécute depuis le dossier `atelier/`.

On importe pandas et les outils de téléchargement, et on prépare le dossier `data/` qui reçoit les fichiers (ignoré par git : la BDIFF brute ne se commite pas, à cause de sa licence, obstacle O04).

In [1]:
import hashlib, io, ssl, zipfile
from pathlib import Path
import certifi, requests
import pandas as pd

DATA = Path("data")
DATA.mkdir(exist_ok=True)
UA = {"User-Agent": "feu-gironde-kpi/0.1 (projet pedagogique)"}

On télécharge les deux fichiers du COG 2026 utiles à la jointure (fiche 06) : les communes au 1er janvier 2026 et les mouvements de communes ; un fichier déjà présent n'est pas retéléchargé.

In [2]:
for f in ["v_commune_2026.csv", "v_mvt_commune_2026.csv"]:
    if not (DATA / f).exists():
        r = requests.get(f"https://www.insee.fr/fr/statistiques/fichier/8740222/{f}", headers=UA, timeout=120)
        r.raise_for_status()
        (DATA / f).write_bytes(r.content)

On exporte la BDIFF (départements 33 et 40, alertes du 01/01/2006 au 31/12/2025) comme dans la fiche 01 : une recherche, puis le zip. Le serveur présente une chaîne TLS incomplète (obstacle O03) : on ajoute au magasin `certifi` l'intermédiaire officiel HARICA, téléchargé à son adresse AIA, et la vérification TLS reste active.
Si l'export échoue, déposer à la main `Incendies.csv` dans `atelier/data/` : la cellule ne le retélécharge pas.

In [3]:
csv = DATA / "Incendies.csv"
if not csv.exists():
    der = requests.get("http://crt.harica.gr/HARICA-GEANT-TLS-R1.cer", timeout=60).content
    ca = DATA / "ca_bdiff.pem"
    ca.write_text(Path(certifi.where()).read_text() + ssl.DER_cert_to_PEM_cert(der))
    s = requests.Session()
    s.headers.update(UA)
    s.verify = str(ca)
    recherche = {"if[dateAlerteDeb][date]": "01/01/2006", "if[dateAlerteFin][date]": "31/12/2025",
                 "if[deprts][value]": "33,40", "if[submit]": ""}
    s.get("https://bdiff.agriculture.gouv.fr/incendies", params=recherche, timeout=300).raise_for_status()
    z = s.get("https://bdiff.agriculture.gouv.fr/incendies/zip", timeout=600)
    z.raise_for_status()
    zipfile.ZipFile(io.BytesIO(z.content)).extract("Incendies.csv", DATA)
print("SHA-256 du CSV :", hashlib.sha256(csv.read_bytes()).hexdigest())

SHA-256 du CSV : 00787e13052709c61cdd3562b4e4fb6be66c4513281e4d345f10eb66557e9854


On lit le CSV de la BDIFF (3 lignes de métadonnées avant l'en-tête, séparateur `;`), en gardant les codes en texte, et on vérifie que le couple (Année, Numéro) identifie bien chaque feu.

In [4]:
b = pd.read_csv(csv, sep=";", skiprows=3, dtype={"Département": str, "Code INSEE": str})
b["date"] = pd.to_datetime(b["Date de première alerte"])
b["ha"] = b["Surface parcourue (m2)"] / 10_000
print(len(b), "feux ; (Année, Numéro) unique :", not b.duplicated(["Année", "Numéro"]).any())

5460 feux ; (Année, Numéro) unique : True


On rapporte chaque code INSEE à sa commune 2026 : un code absent du COG 2026 suit ses mouvements de commune à commune (fusions…), du plus récent, jusqu'à une commune 2026 ; on récupère ensuite le département 2026.

In [5]:
com = pd.read_csv(DATA / "v_commune_2026.csv", dtype=str).query("TYPECOM == 'COM'")[["COM", "DEP"]]
mvt = pd.read_csv(DATA / "v_mvt_commune_2026.csv", dtype=str)
mvt = mvt.query("TYPECOM_AV == 'COM' and TYPECOM_AP == 'COM' and COM_AV != COM_AP and not COM_AV.isin(@com.COM)")
suite = mvt.sort_values("DATE_EFF").drop_duplicates("COM_AV", keep="last").set_index("COM_AV")["COM_AP"]
b["com_2026"] = b["Code INSEE"]
for _ in range(5):  # chaînes de fusions successives
    b["com_2026"] = b["com_2026"].map(suite).fillna(b["com_2026"])
b = b.merge(com.rename(columns={"COM": "com_2026", "DEP": "dep_2026"}), on="com_2026", how="left")

On compte les lignes de Gironde 2021-2025, les codes qui ont changé à la jointure et les lignes perdues avec leur motif (code non rattaché à une commune 2026, ou commune 2026 hors de Gironde).

In [6]:
p = b[b["Année"].between(2021, 2025)].copy()
g = p[p["Département"] == "33"]
print("Lignes Gironde 2021-2025 :", len(g))
print(g.loc[g["Code INSEE"] != g["com_2026"], ["Année", "Code INSEE", "com_2026"]])
print("Perdues, code non rattaché :", g["dep_2026"].isna().sum())
print("Perdues, commune 2026 hors Gironde :", (g["dep_2026"].notna() & (g["dep_2026"] != "33")).sum())

Lignes Gironde 2021-2025 : 571
      Année Code INSEE com_2026
4618   2022      33409    33008
Perdues, code non rattaché : 0
Perdues, commune 2026 hors Gironde : 0


On classe chaque feu en juin ou en juillet-août selon sa date de première alerte, et on écrit une seule fonction de ratio journalier, réutilisée pour K1 et K2 : (valeur de juin ÷ 30) ÷ (valeur de juillet-août ÷ 62) × 100.

In [7]:
p["periode"] = p["date"].dt.month.map({6: "juin", 7: "juil_aout", 8: "juil_aout"})
p["an"] = p["date"].dt.year

def par_an(df, agg):
    t = df.pivot_table(index="an", columns="periode", values="ha", aggfunc=agg).fillna(0)
    t["base"] = t["juil_aout"] / 62
    t["K"] = t["juin"] / 30 / t["base"] * 100
    return t

On calcule K1 pour chaque année en Gironde (territoire 2026) : départs de juin bruts, base = départs de juillet-août ÷ 62, et K1.

In [8]:
gi = p[p["dep_2026"] == "33"]
k1 = par_an(gi, "size")
k1.round(2)

periode,juil_aout,juin,base,K
an,,,,
2021,12,2,0.19,34.44
2022,148,23,2.39,32.12
2023,31,17,0.50,113.33
2024,19,4,0.31,43.51
2025,87,11,1.40,26.13


On calcule la moyenne et la médiane de chaque colonne, puis la part des départs de juin 2021-2025 due à l'année la plus forte.

In [9]:
print(k1[["juin", "base", "K"]].agg(["mean", "median"]).round(2))
print("Année la plus forte :", k1["juin"].idxmax(), "->", round(k1["juin"].max() / k1["juin"].sum() * 100, 1), "% des départs de juin")

periode  juin  base      K
mean     11.4  0.96  49.91
median   11.0  0.50  34.44
Année la plus forte : 2022 -> 40.4 % des départs de juin


Test de sensibilité : on recalcule le K1 médian sans 2024 (année incomplète, règle QR08), puis sur la Gironde et les Landes ensemble.

In [10]:
k1_33_40 = par_an(p[p["dep_2026"].isin(["33", "40"])], "size")
print("Référence :", round(k1["K"].median(), 1))
print("Sans 2024 :", round(k1["K"].drop(2024).median(), 1))
print("Gironde et Landes :", round(k1_33_40["K"].median(), 1))
k1_33_40.round(2)

Référence : 34.4
Sans 2024 : 33.3
Gironde et Landes : 45.2


periode,juil_aout,juin,base,K
an,,,,
2021,24,9,0.39,77.50
2022,209,36,3.37,35.60
2023,65,24,1.05,76.31
2024,32,7,0.52,45.21
2025,162,20,2.61,25.51


On calcule K2, la même formule sur la surface brûlée (ha), médiane 2021-2025, et la variante qui exclut les feux de plus de 1 000 ha (paramètre de la fiche K2).

In [11]:
k2 = par_an(gi, "sum")
k2_sans_grands = par_an(gi[gi["ha"] <= 1000], "sum")
print(pd.DataFrame({"K2": k2["K"], "K2 hors > 1 000 ha": k2_sans_grands["K"]}).round(1))
print("Médianes :", round(k2["K"].median(), 1), "/", round(k2_sans_grands["K"].median(), 1))

        K2  K2 hors > 1 000 ha
an                            
2021   8.4                 8.4
2022   0.2                23.0
2023  63.1                63.1
2024  62.8                62.8
2025   1.7                 1.7
Médianes : 8.4 / 23.0


On calcule K4 : part des feux de plus de 1 ha en juin, moins la même part en juillet-août, en points, puis sa médiane 2021-2025.

In [12]:
k4 = gi.assign(gros=gi["ha"] > 1).pivot_table(index="an", columns="periode", values="gros", aggfunc="mean")
k4["K4"] = (k4["juin"] - k4["juil_aout"]) * 100
print(k4.round(3))
print("K4 médian :", round(k4["K4"].median(), 1), "points")

periode  juil_aout   juin      K4
an                               
2021         0.167  0.000 -16.667
2022         0.236  0.087 -14.953
2023         0.097  0.059  -3.795
2024         0.263  0.250  -1.316
2025         0.230  0.000 -22.989
K4 médian : -15.0 points
